# 🏋️ 03 — YOLOv8 Training
## SmartMine Vision AI · Stage 1: PPE Detection

---

### Objectives
1. Verify the training environment (GPU, CUDA, Ultralytics).
2. Inspect the **dataset configuration YAML**.
3. Understand the **YOLOv8 model selection rationale**.
4. Launch a **reproducible training run**.
5. Visualise **training curves** (loss, mAP) on completion.

---

### Why YOLOv8?
YOLOv8 (Ultralytics, 2023) is the state-of-the-art single-stage detector offering:
- Anchor-free detection head
- C2f bottleneck for richer gradient flow
- Native support for classification, detection, segmentation, and pose
- Built-in augmentation (mosaic, mixup, copy-paste)

### Why YOLOv8n first?
| Variant | Params | COCO mAP | Inference (T4 GPU) |
|---------|--------|----------|--------------------|
| YOLOv8n | 3.2M | 37.3 | 0.99 ms |
| YOLOv8s | 11.2M | 44.9 | 1.20 ms |
| YOLOv8m | 25.9M | 50.2 | 3.52 ms |

Start with **nano** for fast iteration. Upgrade to **small** if mAP < 0.70.

## 1. Setup

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Resolve project root by walking up until 'src/' is found
_cwd = Path().resolve()
PROJECT_ROOT = _cwd
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import torch
import yaml
from ultralytics import YOLO

from src.ppe_detection.utils import (
    CONFIGS_DIR, MODELS_DIR, EXPERIMENTS_DIR, ensure_dirs
)
from src.ppe_detection.trainer import train_ppe_model
ensure_dirs()

## 2. Environment Check

In [ ]:
print("=" * 50)
print("  TRAINING ENVIRONMENT")
print("=" * 50)
print(f"  PyTorch      : {torch.__version__}")

cuda_ok = torch.cuda.is_available()
print(f"  CUDA         : {cuda_ok}")
if cuda_ok:
    print(f"  GPU          : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  VRAM         : {vram:.1f} GB")
    DEVICE = "0"
else:
    import platform
    if platform.system() == "Darwin":
        mps_ok = torch.backends.mps.is_available()
        print(f"  MPS (Apple)  : {mps_ok}")
        DEVICE = "mps" if mps_ok else "cpu"
    else:
        print("  Device       : CPU only")
        DEVICE = "cpu"

print(f"  Selected     : {DEVICE}")
print("=" * 50)

## 3. Dataset Configuration

In [ ]:
DATA_YAML = CONFIGS_DIR / "smartmine_unified.yaml"
print(f"Config: {DATA_YAML}")
print(f"Exists: {DATA_YAML.exists()}")
print()

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

print("DATASET YAML CONTENTS")
print("=" * 50)
print(f"  path  : {cfg['path']}")
print(f"  train : {cfg['train']}")
print(f"  val   : {cfg['val']}")
print(f"  test  : {cfg['test']}")
print(f"  nc    : {cfg['nc']} classes")
print()
print("  Classes:")
for k, v in cfg["names"].items():
    print(f"    {k:2d}: {v}")

## 4. Training Configuration

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Base model | `yolov8n.pt` | COCO pre-trained, fastest iteration |
| Epochs | 100 | Sufficient for fine-tuning a pre-trained model |
| Image size | 640 | Matches dataset pre-processing |
| Batch | -1 (auto) | YOLOv8 auto-selects optimal batch for available VRAM |
| Optimizer | AdamW | Ultralytics default — best convergence on small datasets |
| Augmentation | Built-in | Mosaic, mixup, copy-paste, flips, HSV jitter |
| Patience | 50 | Early stopping if no improvement for 50 epochs |

> **Reproducibility:** All training args are saved automatically to
> `experiments/smartmine_v1/baseline/args.yaml` by Ultralytics.

## 5. Launch Training

In [ ]:
# ⏱  GPU (RTX 3060+): ~20-40 min  |  CPU: several hours
# Reduce epochs to 10 for a quick smoke-test on CPU.

best_weights = train_ppe_model(
    data_yaml  = DATA_YAML,
    base_model = "yolov8n.pt",
    epochs     = 100,
    imgsz      = 640,
    name       = "baseline",
    device     = DEVICE,
)
print(f"\n✅ Training complete.")
print(f"   Best weights → {best_weights}")

## 6. Training Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

EXP_DIR = EXPERIMENTS_DIR / "smartmine_v1" / "baseline"

results_png = EXP_DIR / "results.png"
if results_png.exists():
    img = mpimg.imread(str(results_png))
    plt.figure(figsize=(18, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Training Curves — Loss & mAP", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(str(EXPERIMENTS_DIR / "smartmine_v1" / "training_curves.png"), dpi=150)
    plt.show()
else:
    print(f"Run training first. Expected: {results_png}")

In [ ]:
# Show confusion matrix from training validation
cm_png = EXP_DIR / "confusion_matrix.png"
if cm_png.exists():
    img = mpimg.imread(str(cm_png))
    plt.figure(figsize=(12, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion Matrix — Validation Set", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("Confusion matrix not yet available.")

In [ ]:
# Load and print best metrics from results.csv
import pandas as pd
results_csv = EXP_DIR / "results.csv"
if results_csv.exists():
    results_df = pd.read_csv(results_csv)
    results_df.columns = results_df.columns.str.strip()
    last = results_df.iloc[-1]
    best_epoch = results_df["metrics/mAP50(B)"].idxmax()
    best = results_df.iloc[best_epoch]

    print("FINAL EPOCH METRICS")
    print("=" * 50)
    for col in results_df.columns:
        if "metrics" in col or "loss" in col.lower():
            print(f"  {col:<35}: {last[col]:.4f}")
    print()
    print(f"BEST EPOCH: {int(best_epoch) + 1}")
    print(f"  mAP50    : {best['metrics/mAP50(B)']:.4f}")
    print(f"  mAP50-95 : {best['metrics/mAP50-95(B)']:.4f}")
else:
    print("Results CSV not yet available — run training first.")

## 7. Conclusions & Next Steps

**Training checklist:**
- [ ] Loss curves converging (box_loss, cls_loss, dfl_loss decreasing)
- [ ] mAP50 > 0.70 on validation
- [ ] No obvious overfitting (val loss not rising while train loss falls)
- [ ] Best weights saved to `models/ppe/`

**If mAP50 < 0.70:**
- Try `yolov8s.pt` (more capacity)
- Increase epochs to 150
- Check class imbalance in `01_dataset_exploration.ipynb`

**Next:** `04_evaluation.ipynb` — comprehensive metrics on the held-out test set.